In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

# -----------------------------
# 1) Configuration
# -----------------------------
IMG_SIZE = (256, 256)          # (H, W)
NUM_CLASSES = 4                # example
BATCH_SIZE = 8

# Example file lists (replace with your paths)
train_img_files  = tf.io.gfile.glob("/path/train/images/*.png")
train_mask_files = tf.io.gfile.glob("/path/train/masks/*.png")

val_img_files  = tf.io.gfile.glob("/path/val/images/*.png")
val_mask_files = tf.io.gfile.glob("/path/val/masks/*.png")


# -----------------------------
# 2) Decode helpers
# -----------------------------
def _read_image(path):
    x = tf.io.read_file(path)
    x = tf.image.decode_png(x, channels=3)     # (H,W,3), uint8
    x = tf.image.resize(x, IMG_SIZE, method="bilinear")
    x = tf.cast(x, tf.float32) / 255.0         # float32 in [0,1]
    return x

def _read_mask(path):
    y = tf.io.read_file(path)
    # IMPORTANT: masks should store class IDs as pixel values (0..NUM_CLASSES-1)
    y = tf.image.decode_png(y, channels=1)     # (H,W,1), uint8
    y = tf.image.resize(y, IMG_SIZE, method="nearest")  # keep IDs intact
    y = tf.cast(y, tf.int32)                   # integer class IDs
    return y

def load_pair(img_path, mask_path):
    x = _read_image(img_path)
    y = _read_mask(mask_path)
    return x, y


# -----------------------------
# 3) (Optional) Augmentation
#    Apply SAME spatial transform to image & mask
# -----------------------------
def augment(x, y):
    # random horizontal flip
    do_flip = tf.random.uniform([]) > 0.5
    x = tf.cond(do_flip, lambda: tf.image.flip_left_right(x), lambda: x)
    y = tf.cond(do_flip, lambda: tf.image.flip_left_right(y), lambda: y)
    return x, y


# -----------------------------
# 4) Build tf.data pipelines
# -----------------------------
def make_dataset(img_files, mask_files, training=True):
    ds = tf.data.Dataset.from_tensor_slices((img_files, mask_files))
    if training:
        ds = ds.shuffle(1024, reshuffle_each_iteration=True)
    ds = ds.map(load_pair, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=training)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_img_files, train_mask_files, training=True)
val_ds   = make_dataset(val_img_files,   val_mask_files,   training=False)


# -----------------------------
# 5) Model compile + training
# -----------------------------
# Your segmentation model should output (H,W,NUM_CLASSES) logits or probabilities
# model = ...

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer="adam", loss=loss, metrics=["accuracy"])

model.fit(train_ds, validation_data=val_ds, epochs=20)
